# Sample run of `get_products.py`

This notebook demonstrates the full pipeline of `get_products.py` using `sample.json` — an anonymised subset of real Rohlik order data.

`sample.json` contains 269 orders. Personal information (address, delivery notes, payment details) has been removed, and random products per order have been omitted so the data is not faithful to the original.

**Note:** `get_products.py` fetches product categories from the Rohlik API on the first run. This requires an internet connection and takes a few minutes.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

# VSCode injects __vsc_ipynb_file__ with the notebook's absolute path.
# The notebook lives in Sample/, so the project root is one level up.
try:
    ROOT = Path(__vsc_ipynb_file__).resolve().parent.parent  # notebook is in Sample/
except NameError:
    # Standard Jupyter: cwd is typically the notebook's directory or the launch directory
    ROOT = Path.cwd()
    while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

if not (ROOT / "config.py").exists():
    raise RuntimeError(f"Project root not found. Resolved to: {ROOT}")

os.chdir(ROOT)

## 1. Preview `sample.json`

`sample.json` is an array of 269 orders. Each order has three fields:
- `id` — order identifier
- `orderTime` — date and time of the order
- `items` — list of products bought, each with `id`, `name`, and `amount`

In [2]:
sample_path = ROOT / "data" / "orders" / "sample.json"

with open(sample_path, encoding="utf-8") as f:
    orders = json.load(f)

print(f"Total orders in sample.json: {len(orders)}")
print(f"\nFirst order (id={orders[0]['id']}, {orders[0]['orderTime'][:10]}):")
for item in orders[0]["items"][:5]: #show only first 5 items to avoid flooding the output
    print(f"  x{item['amount']:2d}  {item['name']}")
print(f"  ... ({len(orders[0]['items'])} items total)")

Total orders in sample.json: 269

First order (id=1004626483, 2015-06-07):
  x 4  ARO Mléko trvanlivé polotučné 1,5%
  x 1  CVS Grana Padano 16 měsíců
  x 1  Vodňanské Kuře Originál české párky
  x 4  Kaiserka cereální sypaná lněným semínkem
  x 1  Don Peppe Tvarohové knedlíky jahodové 1kg
  ... (34 items total)


## 2. Run `get_products.py`

`get_products.py` reads all `*.json` (only `sample.json`) files from `data/orders/`. When running for the first time it may take a few minutes.

In [3]:
result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "get_products.py")],
    text=True,
)
if result.returncode != 0:
    print("get_products.py exited with an error.")

Training files not found — running generate_tables.py ...
Loading orders from /home/felipe/Dokumenty/GitHub/Order_Rohlik/data/orders ...
Loaded 8,440 order lines.

Running fetch_categories.py → /home/felipe/Dokumenty/GitHub/Order_Rohlik/data/product_categories ...
  Fetched 2487 (2484 ok, 3 failed).

Found 2487 unique product IDs across all orders.

Fetching mainCategoryId from API (up to 5 parallel requests)...

Saved JSON → /home/felipe/Dokumenty/GitHub/Order_Rohlik/data/product_categories/product_categories.json
Saved CSV  → /home/felipe/Dokumenty/GitHub/Order_Rohlik/data/product_categories/product_categories.csv

Done. 2484 products fetched successfully, 3 failed.


3 product(s) with status != 'ok':
  [not_found] 404 – product no longer exists in the catalogue
     id                                  name    status
1335439             Traumacel Pulvis 2g ster. not_found
1406286 Detritin 2000 IU Vitamin D3 60 tablet not_found
1334451       FAKTU 50MG/G+20MG/G RCT UNG 20G not_found



## 3. Results

The predicted categories are saved to `data/results/products_to_buy.csv`. Each row is one predicted category, with up to three example products and their Rohlik product IDs.

In [4]:
import pandas as pd

results_csv = ROOT / "data" / "results" / "products_to_buy.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df)

,mainCategoryId,example_1,example_2,example_3,id_product_1,id_product_2,id_product_3
0,300114243,Hollandia Selský jogurt švestky se skořicí,Müller Mix Vanilkový jogurt s čokoládovými kul...,Hollandia Selský jogurt jahoda,1343563,1344041,1297329
1,300102002,Žluté tamarillo 1 ks,Mango k dozrání 1 ks,"Rambutan, volně",1409164,1310885,1417516
2,300102010,"Česnek čerstvý, svazek 250 g+","Cibule kuchyňská velká (Ø plodu 80–100 mm), volně","Jarní cibulka, svazek (cca 110 g)",1310733,1442931,1287385
3,300105002,"Miil Čerstvé mléko polotučné 1,5 % tuku","Miil Selské čerstvé plnotučné mléko 3,8 % tuku","Miil Čerstvé mléko plnotučné 3,5%",1406203,1406260,1456220
4,300102012,"Zázvor, volně",Ředkev bílá (cca 500g),"Mrkev s natí, svazek (cca 500 g)",1368371,1302867,1294755
5,300115169,Linteo kuchyňské utěrky bílé XXL 2 vrstvé,Linteo Classic Kuchyňské utěrky 2vrstvé,"Linteo Kuchyňské utěrky bílé 3vrstvé, 3×16 m",1302725,1296499,1379721
6,300102005,Grapefruit malý 1ks,Pomelo červené 1 ks cca 900g,Limeta 1 ks,1294573,1320147,1287619
7,300117325,Kuřecí prsa s kostí od českého výrobce,Prominent Krůtí prsní řízek,Kuřecí prsní řízky od českého výrobce,1425780,1353733,1353879
8,300105054,Schubert Podestýlková vejce čerstvá M,Pohodová vejce velikost L 6ks,Milujeme vejce Čerstvá podestýlková vejce M,1394444,1298113,1453870
9,300123543,Pribináček dezert smetanový,Pribináček kakaový dezert,Pribináček Vanilka dezert,712957,712963,712983
